# Agent State Sandbox
Uses LangGraph's prebuilt `create_react_agent` with a SQLite checkpointer for session memory.

In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

## 1. Imports

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from deps import make_obj
from tool import document_search, web_search

## 2. Prompt + Tools

In [ ]:
AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are DiabeticAssist, a clinical reference chatbot specializing in diabetes care.
You answer questions using verified medical documents only.
Always cite the source document when referencing specific facts.
Never provide personal medical advice or recommend starting/stopping treatments.
Add a safety disclaimer when discussing dosages, insulin regimens, or drug interactions.
Focus on topics including: Type 1 and Type 2 diabetes, prediabetes, gestational diabetes,
blood glucose management, HbA1c targets, medications (metformin, insulin, GLP-1 agonists, SGLT2 inhibitors),
dietary guidance, and diabetes complications.
If a question is outside the scope of available documents, say so clearly.
Only use web_search when the user explicitly asks for the latest or most recent clinical guidelines.
Always call document_search at most once per question."""),
    MessagesPlaceholder("messages"),
])

tools = [document_search, web_search]
tool_node = ToolNode(tools)
llm = AGENT_PROMPT | make_obj().bind_tools(tools)
print("Tools:", [t.name for t in tools])

## 3. Build Agent with Session Memory

In [ ]:
system_prompt = """You are DiabeticAssist, a clinical reference chatbot specializing in diabetes care.
You answer questions using verified medical documents only.
Always cite the source document when referencing specific facts.
Never provide personal medical advice or recommend starting/stopping treatments.
Add a safety disclaimer when discussing dosages, insulin regimens, or drug interactions.
Focus on topics including: Type 1 and Type 2 diabetes, prediabetes, gestational diabetes,
blood glucose management, HbA1c targets, medications (metformin, insulin, GLP-1 agonists, SGLT2 inhibitors),
dietary guidance, and diabetes complications.
If a question is outside the scope of diabetes care, say so clearly and do not call any tool.
Only use web_search when the user explicitly asks for the latest or most recent clinical guidelines.
For all other medical questions, use document_search."""

tools = [document_search, web_search]

agent = create_react_agent(
    model=make_obj(),
    tools=tools,
    prompt=system_prompt,
    checkpointer=InMemorySaver(),
)
print("Agent ready. Tools:", [t.name for t in tools])

## 6. Test — Tool Call Question

In [ ]:
config = {"configurable": {"thread_id": "session-1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="What are the HbA1c targets for type 2 diabetes?")]},
    config=config,
)
print(response["messages"][-1].content)

## 7. Test — No Tool Call Question

In [ ]:
# Same session — agent remembers previous turn
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="What is blue?")]},
    config=config,
)
print(response["messages"][-1].content)

## 8. Test — Metformin Dosage (verifies document_search is always called)

In [ ]:
# New session — fresh context
config2 = {"configurable": {"thread_id": "session-2"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="What is the recommended dosage for metformin in type 2 diabetes?")]},
    config=config2,
)
print(response["messages"][-1].content)